Here's a tiny softmax model's predictions on 10 students, 3 classes (0=Fail, 1=Pass, 2=Distinction):
```
Actual:    [0, 0, 0, 1, 1, 1, 1, 2, 2, 2]
Predicted: [0, 1, 0, 1, 1, 2, 1, 2, 0, 2]
```
Now — in Day 10, "positive" always meant one specific thing: pass/fail, spam/not-spam. There was only ever **one** class that counted as "YES."

Here there are 3 classes. So the question becomes: precision for what, exactly?

The trick is: **pick one class at a time, and pretend it's binary.**

Let's do this for class **1 (Pass)** only, treating "Pass" as the "positive" class and everything else (Fail or Distinction) as "negative":
```
Actual:      [0, 0, 0, 1, 1, 1, 1, 2, 2, 2]
Predicted:   [0, 1, 0, 1, 1, 2, 1, 2, 0, 2]

Is actual=1?    N  N  N  Y  Y  Y  Y  N  N  N
Is pred=1?      N  Y  N  Y  Y  N  Y  N  N  N
```
Look at student index 1: actual=0 (Fail), predicted=1 (Pass). Actual says "not class 1," predicted says "class 1" → that's a **False Positive for class 1.**

Look at index 5: actual=1 (Pass), predicted=2 (Distinction). Actual says "class 1," predicted says "not class 1" → that's a **False Negative for class 1.**

```
index:        0  1  2  3  4  5  6  7  8  9
Actual:       0  0  0  1  1  1  1  2  2  2
Predicted:    0  1  0  1  1  2  1  2  0  2

Is actual=1?  N  N  N  Y  Y  Y  Y  N  N  N
Is pred=1?    N  Y  N  Y  Y  N  Y  N  N  N
```
Line them up position by position and only count where both are Y:
```
index 3: actual=Y, pred=Y → TP
index 4: actual=Y, pred=Y → TP
index 5: actual=Y, pred=N (predicted 2, not 1) → this is a miss, not a TP
index 6: actual=Y, pred=Y → TP
```

class 1's full confusion matrix (FP, FN, TN)
```
index:        0  1  2  3  4  5  6  7  8  9
Actual:       0  0  0  1  1  1  1  2  2  2
Predicted:    0  1  0  1  1  2  1  2  0  2

Is actual=1?  N  N  N  Y  Y  Y  Y  N  N  N
Is pred=1?    N  Y  N  Y  Y  N  Y  N  N  N
```
```
TP (actual=Y, pred=Y): indices 3, 4, 6 → TP = 3
FP (actual=N, pred=Y): index 1 → FP = 1
FN (actual=Y, pred=N): index 5 → FN = 1
TN (actual=N, pred=N): indices 0, 2, 7, 8, 9 → TN = 5
```
**Check: 3+1+1+5 = 10 ✓**

So for class 1 alone:
```
Precision₁ = TP/(TP+FP) = 3/(3+1) = 0.75
Recall₁    = TP/(TP+FN) = 3/(3+1) = 0.75
F1₁        = 2×(0.75×0.75)/(0.75+0.75) = 0.75
```

class 2's full confusion matrix (FP, FN, TN)
```
index:        0  1  2  3  4  5  6  7  8  9
Actual:       0  0  0  1  1  1  1  2  2  2
Predicted:    0  1  0  1  1  2  1  2  0  2
Is actual=2?  N  N  N  N  N  N  N  Y  Y  Y
Is pred=2?    N  N  N  N  N  Y  N  Y  N  Y
```
```
TP (Y,Y): indices 7, 9 → TP = 2 ✓
FP (N,Y): index 5 → FP = 1
FN (Y,N): index 8 → FN = 1
TN (N,N): indices 0,1,2,3,4,6 → TN = 6
```
**Check: 2+1+1+6 = 10 ✓**
```
Precision₂ = 2/(2+1) = 0.667
Recall₂    = 2/(2+1) = 0.667
F1₂        = 0.667
```
And for completeness, class 0 (Fail) — actual=0 at indices 0,1,2; predicted=0 at indices 0,2,8:
```
TP = 2 (indices 0,2)   FP = 1 (index 8)   FN = 1 (index 1)   TN = 6
Precision₀ = 0.667   Recall₀ = 0.667   F1₀ = 0.667
```
Now we have 3 separate scores per class:
```
Class	      Precision	Recall	F1
0 (Fail)	0.667	0.667	0.667
1 (Pass)	0.75	0.75	0.75
2 (Distinction)	0.667	0.667	0.667
```

just add them up and divide by 3, like averaging test scores.
```
Macro F1 = (0.667 + 0.75 + 0.667) / 3 = 0.695
```
That's exactly what macro-averaging is — treat every class as equally important, average their scores straight, no weighting by how many students are actually in each class.

But here's the catch, and it's the whole reason 3 different averaging schemes exist. Look back at our data:
```
Actual: [0, 0, 0, 1, 1, 1, 1, 2, 2, 2]
```
Class 0 has 3 students, class 1 has 4 students, class 2 has 3 students — fairly balanced here. But imagine instead a dataset like this:
```
Actual: [0]*95 + [1]*3 + [2]*2     # 95 Fails, 3 Passes, 2 Distinctions
```
If the model does great on class 0 (F1=0.95, because it's 95% of the data and easy to get right) but terribly on classes 1 and 2 (F1=0.10 each, because there's barely any data to learn from):
```
Macro F1 = (0.95 + 0.10 + 0.10) / 3 = 0.383
```
Macro-average punishes the model hard for doing badly on the rare classes — even though those rare classes barely matter to overall accuracy. Is that punishment fair, or unfair? It depends entirely on your use case — and that's exactly the design question micro and weighted averaging answer differently.

If the model gets 95/95 of the Fail students right, plus a couple right on the tiny classes, real-world accuracy would look something like:
```
Correct ≈ 95 (class 0) + ~0.3 (class 1, 10% of 3) + ~0.2 (class 2, 10% of 2)
        ≈ 95.5 out of 100
Accuracy ≈ 95.5%
```
That's a huge, reassuring number — "the model is right 95% of the time!"

**But Macro F1 = 0.383 (38.3%) — a terrible-looking score for the exact same model.**

So macro-average makes the model look worse, not better, compared to what raw accuracy suggests. Here's why: macro-average treats the 2-student Distinction class as just as important as the 95-student Fail class when averaging — so the model's near-total failure on the tiny classes drags the average way down, even though those failures barely touch real-world accuracy.

**This is the actual point of macro-average, and why it exists:** it deliberately refuses to let a model hide poor minority-class performance behind a big, easy majority class.

Micro-averaging works differently from macro. Instead of computing F1 per class and then averaging, **it pools all the TP, FP, FN** counts across all classes first, then computes one single precision/recall/F1 from those pooled totals.

Using our small 10-student example (not the imbalanced hypothetical), here are the per-class counts we already computed:
```
Class	TP	FP	FN
0	2	1	1
1	3	1	1
2	2	1	1
```
**Micro-average pools them:**
```
Total TP = 2 + 3 + 2 = 7
Total FP = 1 + 1 + 1 = 3
Total FN = 1 + 1 + 1 = 3

Micro Precision = 7 / (7+3) = 0.70
Micro Recall    = 7 / (7+3) = 0.70
Micro F1        = 0.70
```
Now here's the key insight: **every wrong prediction is exactly one FP for the wrong class AND one FN for the right class** — so total FP always equals total FN in standard multiclass (each mistake contributes exactly one of each). That's why Micro Precision = Micro Recall = Micro F1 always, and why it equals plain accuracy: **7 correct out of 10 total = 70% = our accuracy here.**

Back to your imbalanced 95/3/2 example: since the 95-student class dominates the pooled totals, **Micro F1 would land almost exactly at accuracy's ~95.5%** — nowhere near macro's harsh 0.383. Micro lets the huge class's success drown out the tiny classes' failures, because it's counting predictions, not classes.

So now we have two honest but opposite stories about the same model:
```
**Macro F1 (0.383):** "this model is bad — it fails badly on 2 of 3 classes"
**Micro F1 (≈0.955):** "this model is great — it's right 95%+ of the time"
```

**Weighted average** takes macro's approach (average the per-class F1 scores, not pooled counts) but instead of treating every class equally, it weights each class's F1 by **how many actual students are really in that class** — called its "support."

Using our small 10-student example, the support (actual count) per class was:
```
Class 0: 3 students
Class 1: 4 students
Class 2: 3 students
Total: 10
```
```
Weighted F1 = (F1₀ × support₀ + F1₁ × support₁ + F1₂ × support₂) / total_support
            = (0.667×3 + 0.75×4 + 0.667×3) / 10
            = (2.001 + 3.0 + 2.001) / 10
            = 7.002 / 10
            = 0.700
```
Notice: in this particular case, Weighted F1 (0.700) happens to land very close to Micro F1 (0.700) — because our classes are fairly balanced (3, 4, 3), so weighting by support doesn't shift things much from a plain average.

But go back to the 95/3/2 imbalanced example: weighted-average would multiply class 0's good F1 (0.95) by its huge weight (95), and multiply classes 1/2's terrible F1s (0.10 each) by their tiny weights (3 and 2). The result lands close to accuracy again (like micro) — but it's still computed as a genuine per-class average, not pooled counts, which matters in edge cases like multi-label problems where micro and weighted actually diverge.

**So here's the real difference in one line, worth locking in:**

**Macro** = "treat every class equally, no matter how rare" → exposes minority-class failure

**Weighted** = "treat every class fairly by its real-world frequency" → honest average, still per-class

**Micro** = "just count every single prediction, pooled" → equals accuracy in standard multiclass

Let's lock all three in with the formulas side by side, then move to building this in code.
```
Macro    = mean of per-class F1 scores, unweighted
Weighted = mean of per-class F1 scores, weighted by support (actual count per class)
Micro    = pool all TP/FP/FN across classes first, THEN compute one F1
         = equals accuracy in standard (single-label) multiclass
```

**Mechanically, here's what class_weight='balanced' actually does** — and it's a direct extension of Day 9's Binary Cross-Entropy:
```
Normal loss:    each sample contributes equally to the loss
Balanced loss:  each sample's contribution is multiplied by a weight,
                inversely proportional to its class's frequency

weight_for_class_c = n_samples / (n_classes × count_of_class_c)
```
A rare class (class 2, only 6 training samples) gets a **huge weight multiplier**, so every mistake on it costs the loss function much more than a mistake on the 210-sample majority class. This is the exact same idea as Day 7's Ridge penalty (+λwᵀw inside the loss) — except here the "tax" targets misclassifying rare-class samples instead of large weights.

### Confusion Matrix

In [3]:
import numpy as np

y_actual = np.array([0, 0, 0, 1, 1, 1, 1, 2, 2, 2])
y_pred   = np.array([0, 1, 0, 1, 1, 2, 1, 2, 0, 2])

n_classes = 3
conf_matrix = np.zeros((n_classes, n_classes), dtype=int)

for actual, pred in zip(y_actual, y_pred):
    conf_matrix[actual, pred] += 1

print("Confusion Matrix (rows=actual, cols=predicted):")
print(conf_matrix)

tp = np.diag(conf_matrix)
fp = conf_matrix.sum(axis=0) - tp
fn = conf_matrix.sum(axis=1) - tp

print("\nPer-class TP:", tp)
print("Per-class FP:", fp)
print("Per-class FN:", fn)

Confusion Matrix (rows=actual, cols=predicted):
[[2 1 0]
 [0 3 1]
 [1 0 2]]

Per-class TP: [2 3 2]
Per-class FP: [1 1 1]
Per-class FN: [1 1 1]


#### Full Confusion Matrix

In [5]:
import numpy as np

y_actual = np.array([0, 0, 0, 1, 1, 1, 1, 2, 2, 2])
y_pred   = np.array([0, 1, 0, 1, 1, 2, 1, 2, 0, 2])

n_classes = 3
conf_matrix = np.zeros((n_classes, n_classes), dtype=int)

for actual, pred in zip(y_actual, y_pred):
    conf_matrix[actual, pred] += 1

print("Confusion Matrix (rows=actual, cols=predicted):")
print(conf_matrix)

tp = np.diag(conf_matrix)
fp = conf_matrix.sum(axis=0) - tp
fn = conf_matrix.sum(axis=1) - tp

print("\nPer-class TP:", tp)
print("Per-class FP:", fp)
print("Per-class FN:", fn)

precision_per_class = tp / (tp + fp)
recall_per_class = tp / (tp + fn)
f1_per_class = 2 * (precision_per_class * recall_per_class) / (precision_per_class + recall_per_class)

print("\nPer-class Precision:", np.round(precision_per_class, 4))
print("Per-class Recall:   ", np.round(recall_per_class, 4))
print("Per-class F1:       ", np.round(f1_per_class, 4))

# MACRO: plain mean across classes
macro_precision = precision_per_class.mean()
macro_recall = recall_per_class.mean()
macro_f1 = f1_per_class.mean()

# MICRO: pool TP/FP/FN first, then compute once
micro_tp, micro_fp, micro_fn = tp.sum(), fp.sum(), fn.sum()
micro_precision = micro_tp / (micro_tp + micro_fp)
micro_recall = micro_tp / (micro_tp + micro_fn)
micro_f1 = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall)

# WEIGHTED: mean weighted by support (row totals = actual count per class)
support = conf_matrix.sum(axis=1)
weighted_precision = (precision_per_class * support).sum() / support.sum()
weighted_recall = (recall_per_class * support).sum() / support.sum()
weighted_f1 = (f1_per_class * support).sum() / support.sum()

print("\nSupport per class:", support)
print(f"\nMacro    -> P: {macro_precision:.4f}  R: {macro_recall:.4f}  F1: {macro_f1:.4f}")
print(f"Micro    -> P: {micro_precision:.4f}  R: {micro_recall:.4f}  F1: {micro_f1:.4f}")
print(f"Weighted -> P: {weighted_precision:.4f}  R: {weighted_recall:.4f}  F1: {weighted_f1:.4f}")

Confusion Matrix (rows=actual, cols=predicted):
[[2 1 0]
 [0 3 1]
 [1 0 2]]

Per-class TP: [2 3 2]
Per-class FP: [1 1 1]
Per-class FN: [1 1 1]

Per-class Precision: [0.6667 0.75   0.6667]
Per-class Recall:    [0.6667 0.75   0.6667]
Per-class F1:        [0.6667 0.75   0.6667]

Support per class: [3 4 3]

Macro    -> P: 0.6944  R: 0.6944  F1: 0.6944
Micro    -> P: 0.7000  R: 0.7000  F1: 0.7000
Weighted -> P: 0.7000  R: 0.7000  F1: 0.7000


### Verify via Sklearn

In [6]:
from sklearn.metrics import confusion_matrix, f1_score

sk_conf_matrix = confusion_matrix(y_actual, y_pred)
sk_macro_f1 = f1_score(y_actual, y_pred, average='macro')
sk_micro_f1 = f1_score(y_actual, y_pred, average='micro')
sk_weighted_f1 = f1_score(y_actual, y_pred, average='weighted')

print("\n--- sklearn check ---")
print("sklearn confusion matrix matches ours:", np.array_equal(sk_conf_matrix, conf_matrix))
print(f"Macro F1    -> ours: {macro_f1:.4f}  sklearn: {sk_macro_f1:.4f}  match: {np.isclose(macro_f1, sk_macro_f1)}")
print(f"Micro F1    -> ours: {micro_f1:.4f}  sklearn: {sk_micro_f1:.4f}  match: {np.isclose(micro_f1, sk_micro_f1)}")
print(f"Weighted F1 -> ours: {weighted_f1:.4f}  sklearn: {sk_weighted_f1:.4f}  match: {np.isclose(weighted_f1, sk_weighted_f1)}")


--- sklearn check ---
sklearn confusion matrix matches ours: True
Macro F1    -> ours: 0.6944  sklearn: 0.6944  match: True
Micro F1    -> ours: 0.7000  sklearn: 0.7000  match: True
Weighted F1 -> ours: 0.7000  sklearn: 0.7000  match: True


#### class imbalance / class_weight='balanced'

In [8]:
# Block 1 — build the imbalanced dataset with real features:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report

np.random.seed(42)

# 3-class imbalanced data with 2 features
# Class 0 (majority): 300 samples, centered at (0,0)
# Class 1 (minority): 15 samples, centered at (3,3)
# Class 2 (rare minority): 8 samples, centered at (-3,3)
X0 = np.random.randn(300, 2) * 1.5 + [0, 0]
X1 = np.random.randn(15, 2) * 1.0 + [3, 3]
X2 = np.random.randn(8, 2) * 1.0 + [-3, 3]

X = np.vstack([X0, X1, X2])
y = np.array([0]*300 + [1]*15 + [2]*8)

print("Class distribution:", np.bincount(y))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Train class counts:", np.bincount(y_train))
print("Test class counts: ", np.bincount(y_test))

Class distribution: [300  15   8]
Train class counts: [210  10   6]
Test class counts:  [90  5  2]


In [9]:
# Block 2 — train the default (unweighted) model and evaluate:

# --- Model 1: default, no class weighting ---
model_default = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred_default = model_default.predict(X_test)

print("\n=== DEFAULT (unweighted) ===")
print("Confusion matrix:\n", confusion_matrix(y_test, pred_default))
print(classification_report(y_test, pred_default, digits=3, zero_division=0))
print(f"Macro F1: {f1_score(y_test, pred_default, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, pred_default, average='weighted'):.4f}")


=== DEFAULT (unweighted) ===
Confusion matrix:
 [[90  0  0]
 [ 2  3  0]
 [ 1  0  1]]
              precision    recall  f1-score   support

           0      0.968     1.000     0.984        90
           1      1.000     0.600     0.750         5
           2      1.000     0.500     0.667         2

    accuracy                          0.969        97
   macro avg      0.989     0.700     0.800        97
weighted avg      0.970     0.969     0.965        97

Macro F1: 0.8001
Weighted F1: 0.9650


In [10]:
# Block 3 — train with class_weight='balanced' and compare both models side by side:

# --- Model 2: class_weight='balanced' ---
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train, y_train)
pred_balanced = model_balanced.predict(X_test)

print("\n=== BALANCED (class_weight='balanced') ===")
print("Confusion matrix:\n", confusion_matrix(y_test, pred_balanced))
print(classification_report(y_test, pred_balanced, digits=3, zero_division=0))
print(f"Macro F1: {f1_score(y_test, pred_balanced, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, pred_balanced, average='weighted'):.4f}")

print("\n=== COMPARISON ===")
print(f"Macro F1:    default={f1_score(y_test, pred_default, average='macro'):.4f}  "
      f"balanced={f1_score(y_test, pred_balanced, average='macro'):.4f}")
print(f"Weighted F1: default={f1_score(y_test, pred_default, average='weighted'):.4f}  "
      f"balanced={f1_score(y_test, pred_balanced, average='weighted'):.4f}")


=== BALANCED (class_weight='balanced') ===
Confusion matrix:
 [[88  1  1]
 [ 0  5  0]
 [ 0  0  2]]
              precision    recall  f1-score   support

           0      1.000     0.978     0.989        90
           1      0.833     1.000     0.909         5
           2      0.667     1.000     0.800         2

    accuracy                          0.979        97
   macro avg      0.833     0.993     0.899        97
weighted avg      0.985     0.979     0.981        97

Macro F1: 0.8993
Weighted F1: 0.9808

=== COMPARISON ===
Macro F1:    default=0.8001  balanced=0.8993
Weighted F1: default=0.9650  balanced=0.9808
